# Notebook 2: Pipeline Completa ARIMA/SARIMA

Este notebook implementa toda a **pipeline de modelagem de séries temporais** estudada nas aulas, seguindo o fluxo:

## Pipeline:
1. **Carregamento dos Dados**
2. **Análise Exploratória**: visualização, estatísticas descritivas
3. **Testes de Estacionariedade**: ADF e KPSS
4. **Diferenciação**: transformar série não estacionária
5. **Decomposição Sazonal** (se aplicável): STL
6. **Identificação de Ordem**: ACF e PACF
7. **Modelagem ARIMA**: busca de melhores parâmetros (p, d, q)
8. **Modelagem SARIMA**: busca de melhores parâmetros (p, d, q)(P, D, Q, m)
9. **Diagnóstico de Resíduos**: teste de Ljung-Box, análise ACF
10. **Previsão e Avaliação**: métricas de erro

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import kpss, adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Configurações
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (20, 6)
pd.set_option('display.max_columns', None)

## Funções Utilitárias

Funções reutilizáveis para análise de séries temporais.

In [ ]:
def autocovariance(series, lag):
    """
    Calcula a autocovariância para um dado lag.
    """
    n = len(series)
    mean = np.mean(series)
    return np.sum((series[lag:] - mean) * (series[:n - lag] - mean)) / n

def autocorrelation(series, lag):
    """
    Calcula a autocorrelação para um dado lag.
    """
    return autocovariance(series, lag) / autocovariance(series, 0)

def analisar_estacionariedade(serie_raw, col, titulo, max_lag=20):
    """
    Função completa de análise de estacionariedade.
    Plota a série, ACF, PACF e exibe resultados de ADF e KPSS.
    
    Parâmetros:
    -----------
    serie_raw : DataFrame
        DataFrame contendo a série
    col : str
        Nome da coluna numérica
    titulo : str
        Título para os gráficos
    max_lag : int
        Número máximo de lags para ACF/PACF
    
    Retorna:
    --------
    dict : Resultados dos testes
    """
    valores = serie_raw[col].dropna().values

    # Plot da série, ACF e PACF
    fig, axes = plt.subplots(1, 3, figsize=(20, 4))
    axes[0].plot(valores)
    axes[0].set_title(titulo + ' - Série', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('tempo')
    axes[0].set_ylabel('valor')
    axes[0].grid(True, alpha=0.3)
    
    plot_acf(valores, lags=max_lag, ax=axes[1], title=titulo + ' - ACF')
    plot_pacf(valores, lags=max_lag, ax=axes[2], title=titulo + ' - PACF')
    
    plt.tight_layout()
    plt.show()

    sep = '=' * 55

    # ADF Test
    adf_res = adfuller(valores)
    print('\n' + sep)
    print('  ' + titulo)
    print(sep)
    print('\n[ADF - H0: série NÃO é estacionária]')
    print('  Estatística ADF : ' + str(round(adf_res[0], 4)))
    print('  p-valor         : ' + str(round(adf_res[1], 4)))
    print('  Valores Críticos:')
    for chave, valor in adf_res[4].items():
        print('    ' + chave + ': ' + str(round(valor, 4)))
    adf_estac = adf_res[1] < 0.05
    sinal_adf = '< 0.05' if adf_estac else '>= 0.05'
    concl_adf = 'ESTACIONÁRIA' if adf_estac else 'NÃO ESTACIONÁRIA'
    print('  → Conclusão ADF: ' + concl_adf + ' (p ' + sinal_adf + ')')

    # KPSS Test
    kpss_res = kpss(valores, regression='c')
    print('\n[KPSS - H0: série É estacionária]')
    print('  Estatística KPSS: ' + str(round(kpss_res[0], 4)))
    print('  p-valor         : ' + str(round(kpss_res[1], 4)))
    print('  Valores Críticos:')
    for chave, valor in kpss_res[3].items():
        print('    ' + chave + ': ' + str(round(valor, 4)))
    kpss_estac = kpss_res[1] >= 0.05
    sinal_kpss = '>= 0.05' if kpss_estac else '< 0.05'
    concl_kpss = 'ESTACIONÁRIA' if kpss_estac else 'NÃO ESTACIONÁRIA'
    print('  → Conclusão KPSS: ' + concl_kpss + ' (p ' + sinal_kpss + ')')

    # Veredicto Final
    print('\n[VEREDICTO FINAL]')
    if adf_estac and kpss_estac:
        veredicto = 'ESTACIONÁRIA (ambos os testes concordam)'
    elif not adf_estac and not kpss_estac:
        veredicto = 'NÃO ESTACIONÁRIA (ambos os testes concordam)'
    elif adf_estac and not kpss_estac:
        veredicto = 'INCONCLUSIVO - possível diferença estacionária (DS)'
    else:
        veredicto = 'INCONCLUSIVO - possível estacionariedade em torno de tendência (TS)'
    print('  ' + veredicto + '\n')
    
    return {
        'adf_pval': adf_res[1], 
        'kpss_pval': kpss_res[1],
        'adf_estac': adf_estac, 
        'kpss_estac': kpss_estac, 
        'veredicto': veredicto
    }

def calcular_metricas(y_true, y_pred):
    """
    Calcula métricas de erro (MAE, RMSE, MAPE).
    """
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true_clean = np.array(y_true)[mask]
    y_pred_clean = np.array(y_pred)[mask]
    
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    mape = mean_absolute_percentage_error(y_true_clean, y_pred_clean) * 100
    
    return {'MAE': round(mae, 4), 'RMSE': round(rmse, 4), 'MAPE': round(mape, 4)}

---
## 1. Carregamento dos Dados

Carregue sua série temporal (CSV ou Excel).

In [ ]:
# ===== CONFIGURAÇÃO: AJUSTE PARA SUA BASE =====

# Carregar de arquivo
# df = pd.read_csv('sua_serie.csv')
# df = pd.read_excel('sua_serie.xlsx')

# Exemplo para demonstração
df = pd.DataFrame({
    'data': pd.date_range('2020-01-01', periods=200, freq='D'),
    'valor': np.random.randn(200).cumsum() + 100  # Série com tendência
})

# DEFINA O NOME DA COLUNA DE VALORES
coluna_valor = 'valor'

print(f"Shape: {df.shape}")
print(f"\nPrimeiras linhas:")
display(df.head())
print(f"\nÚltimas linhas:")
display(df.tail())

---
## 2. Análise Exploratória

Visualização inicial e estatísticas descritivas.

In [ ]:
# Estatísticas descritivas
print("Estatísticas Descritivas:")
print(df[coluna_valor].describe())

# Visualização da série completa
fig, axes = plt.subplots(2, 1, figsize=(20, 8))

# Série temporal
axes[0].plot(df[coluna_valor].values, linewidth=2)
axes[0].set_title('Série Temporal Original', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tempo')
axes[0].set_ylabel('Valor')
axes[0].grid(True, alpha=0.3)

# Histograma
axes[1].hist(df[coluna_valor].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[1].set_title('Distribuição dos Valores', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Valor')
axes[1].set_ylabel('Frequência')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Testes de Estacionariedade

Aplicar testes ADF e KPSS para verificar se a série é estacionária.

In [ ]:
# Teste de estacionariedade na série original
resultado_original = analisar_estacionariedade(df, coluna_valor, 'Série Original')

---
## 4. Diferenciação

Se a série não for estacionária, aplicar diferenciação.

**Diferenciação de Ordem 1**: $y'_t = y_t - y_{t-1}$

**Diferenciação de Ordem 2**: $y''_t = y'_t - y'_{t-1}$

In [ ]:
# Aplicar diferenciação se necessário
if not resultado_original['adf_estac'] or not resultado_original['kpss_estac']:
    print("\n⚠️  Série NÃO ESTACIONÁRIA detectada. Aplicando diferenciação...\n")
    
    # Diferenciação de ordem 1
    df['diff_1'] = df[coluna_valor].diff()
    
    # Teste de estacionariedade na série diferenciada
    df_diff = df.dropna()
    resultado_diff1 = analisar_estacionariedade(df_diff, 'diff_1', 'Série Diferenciada (d=1)')
    
    # Se ainda não estacionária, aplicar segunda diferenciação
    if not resultado_diff1['adf_estac'] or not resultado_diff1['kpss_estac']:
        print("\n⚠️  Série ainda não estacionária. Aplicando diferenciação d=2...\n")
        df['diff_2'] = df['diff_1'].diff()
        df_diff2 = df.dropna()
        resultado_diff2 = analisar_estacionariedade(df_diff2, 'diff_2', 'Série Diferenciada (d=2)')
        ordem_d = 2
    else:
        ordem_d = 1
        print("\n✅ Série estacionária após diferenciação d=1\n")
else:
    ordem_d = 0
    print("\n✅ Série já é ESTACIONÁRIA (d=0)\n")

---
## 5. Decomposição Sazonal (STL)

Analisa tendência, sazonalidade e resíduos separadamente.

In [ ]:
# ===== CONFIGURAÇÃO: AJUSTE O PERÍODO SAZONAL =====
# Período sazonal (m):
#   - Dados diários com padrão semanal: m=7
#   - Dados mensais com padrão anual: m=12
#   - Sem sazonalidade: pule esta etapa

periodo_sazonal = 7  # AJUSTE CONFORME SUA SÉRIE

print(f"Decomposição STL com período sazonal m={periodo_sazonal}\n")

# Aplicar STL
stl = STL(df[coluna_valor], period=periodo_sazonal, robust=True).fit()

# Visualização da decomposição
fig = stl.plot()
fig.set_size_inches(20, 10)
plt.tight_layout()
plt.show()

# Força da sazonalidade
forca_sazonalidade = 1 - (np.var(stl.resid) / np.var(df[coluna_valor]))
print(f"\n📊 Força da Sazonalidade: {forca_sazonalidade:.4f}")

if forca_sazonalidade > 0.6:
    print("   → Sazonalidade FORTE detectada. Recomenda-se usar SARIMA.")
elif forca_sazonalidade > 0.3:
    print("   → Sazonalidade MODERADA detectada. Considere usar SARIMA.")
else:
    print("   → Sazonalidade FRACA. ARIMA pode ser suficiente.")

---
## 6. Divisão Treino/Teste

Separar dados para validação do modelo.

In [ ]:
# ===== CONFIGURAÇÃO: HORIZONTE DE PREVISÃO =====
h = 10  # Número de períodos para teste

train = df[coluna_valor].iloc[:-h]
test = df[coluna_valor].iloc[-h:]

print(f"Tamanho do conjunto de treino: {len(train)}")
print(f"Tamanho do conjunto de teste:  {len(test)}")
print(f"\nÚltimos valores de treino:")
print(train.tail())
print(f"\nValores de teste:")
print(test)

---
## 7. Modelagem ARIMA

Busca automática dos melhores parâmetros (p, d, q) usando BIC.

**ARIMA(p, d, q)**:
- **p**: ordem do componente AR (AutoRegressive)
- **d**: ordem de diferenciação
- **q**: ordem do componente MA (Moving Average)

In [ ]:
# Grid search para ARIMA
print("🔍 Buscando melhor modelo ARIMA...\n")

melhor_bic = float("inf")
melhor_ordem = None
melhor_modelo = None
historico_arima = []

# Range de busca
p_range = range(0, 5)
d_range = range(0, 3)
q_range = range(0, 5)

for p in p_range:
    for d in d_range:
        for q in q_range:
            if p == 0 and d == 0 and q == 0:
                continue
            try:
                modelo = ARIMA(train, order=(p, d, q)).fit()
                bic = float(modelo.bic)
                historico_arima.append({'p': p, 'd': d, 'q': q, 'BIC': round(bic, 2)})
                
                if bic < melhor_bic:
                    melhor_bic = bic
                    melhor_ordem = (p, d, q)
                    melhor_modelo = modelo
            except:
                continue

print(f"\n✅ Melhor ARIMA{melhor_ordem} com BIC = {melhor_bic:.2f}\n")
print(melhor_modelo.summary())

# Top 5 modelos
df_hist_arima = pd.DataFrame(historico_arima).sort_values('BIC').head(10)
print("\n📋 Top 10 Modelos ARIMA (menor BIC):")
display(df_hist_arima)

---
## 8. Diagnóstico de Resíduos (ARIMA)

Verificar se os resíduos são ruído branco (não correlacionados).

In [ ]:
# Análise de resíduos
res_arima = melhor_modelo.resid

# Teste de Ljung-Box
lb = acorr_ljungbox(res_arima, lags=[20])
print("Teste Ljung-Box (lags=20):")
print(lb)
print(f"\np-value = {lb['lb_pvalue'].values[0]:.4f}")
if lb['lb_pvalue'].values[0] > 0.05:
    print("✅ Resíduos são ruído branco (não há autocorrelação significativa)")
else:
    print("⚠️  Resíduos ainda possuem autocorrelação")

# Plotar ACF dos resíduos
plt.figure(figsize=(15, 4))
plot_acf(res_arima, lags=20)
plt.title("ACF dos Resíduos - ARIMA", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Previsão ARIMA

Gerar previsões e calcular métricas de erro.

In [ ]:
# Fazer previsões
previsoes_arima = melhor_modelo.forecast(steps=h)

# Calcular métricas
metricas_arima = calcular_metricas(test.values, previsoes_arima)
print("\n📊 Métricas ARIMA:")
for metrica, valor in metricas_arima.items():
    print(f"   {metrica}: {valor}")

# Visualização
plt.figure(figsize=(20, 6))
plt.plot(range(len(train)), train.values, label='Treino', linewidth=2)
plt.plot(range(len(train), len(train) + len(test)), test.values, 
         label='Teste (Real)', marker='o', linewidth=2, markersize=8)
plt.plot(range(len(train), len(train) + len(test)), previsoes_arima, 
         label=f'Previsão ARIMA{melhor_ordem}', marker='x', linewidth=2, 
         markersize=10, linestyle='--')
plt.axvline(x=len(train), color='red', linestyle=':', linewidth=2, label='Início do Teste')
plt.title(f'Previsão ARIMA{melhor_ordem}', fontsize=14, fontweight='bold')
plt.xlabel('Tempo')
plt.ylabel('Valor')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Modelagem SARIMA

Busca automática dos melhores parâmetros (p, d, q)(P, D, Q, m).

**SARIMA(p, d, q)(P, D, Q, m)**:
- **(p, d, q)**: componentes não sazonais (como ARIMA)
- **(P, D, Q)**: componentes sazonais
- **m**: período sazonal

In [ ]:
# Grid search para SARIMA
print("🔍 Buscando melhor modelo SARIMA...\n")

melhor_bic_sarima = float("inf")
melhor_ordem_sarima = None
melhor_modelo_sarima = None
historico_sarima = []

# Range de busca (reduzido para performance)
p_range = range(0, 3)
d_range = range(0, 2)
q_range = range(0, 3)
P_range = range(0, 2)
D_range = range(0, 2)
Q_range = range(0, 2)
m = periodo_sazonal  # Definido anteriormente

print(f"Período sazonal (m): {m}\n")

for p in p_range:
    for d in d_range:
        for q in q_range:
            for P in P_range:
                for D in D_range:
                    for Q in Q_range:
                        if p == 0 and d == 0 and q == 0 and P == 0 and D == 0 and Q == 0:
                            continue
                        try:
                            modelo = SARIMAX(train, 
                                           order=(p, d, q),
                                           seasonal_order=(P, D, Q, m)).fit(disp=False)
                            bic = float(modelo.bic)
                            historico_sarima.append({
                                'p': p, 'd': d, 'q': q,
                                'P': P, 'D': D, 'Q': Q, 'm': m,
                                'BIC': round(bic, 2)
                            })
                            
                            if bic < melhor_bic_sarima:
                                melhor_bic_sarima = bic
                                melhor_ordem_sarima = ((p, d, q), (P, D, Q, m))
                                melhor_modelo_sarima = modelo
                        except:
                            continue

print(f"\n✅ Melhor SARIMA{melhor_ordem_sarima[0]}x{melhor_ordem_sarima[1]} com BIC = {melhor_bic_sarima:.2f}\n")
print(melhor_modelo_sarima.summary())

# Top 10 modelos
df_hist_sarima = pd.DataFrame(historico_sarima).sort_values('BIC').head(10)
print("\n📋 Top 10 Modelos SARIMA (menor BIC):")
display(df_hist_sarima)

---
## 11. Diagnóstico de Resíduos (SARIMA)

In [ ]:
# Análise de resíduos SARIMA
res_sarima = melhor_modelo_sarima.resid

# Teste de Ljung-Box
lb_sarima = acorr_ljungbox(res_sarima, lags=[20])
print("Teste Ljung-Box (lags=20):")
print(lb_sarima)
print(f"\np-value = {lb_sarima['lb_pvalue'].values[0]:.4f}")
if lb_sarima['lb_pvalue'].values[0] > 0.05:
    print("✅ Resíduos são ruído branco (não há autocorrelação significativa)")
else:
    print("⚠️  Resíduos ainda possuem autocorrelação")

# Plotar ACF dos resíduos
plt.figure(figsize=(15, 4))
plot_acf(res_sarima, lags=20)
plt.title("ACF dos Resíduos - SARIMA", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 12. Previsão SARIMA

In [ ]:
# Fazer previsões SARIMA
previsoes_sarima = melhor_modelo_sarima.forecast(steps=h)

# Calcular métricas
metricas_sarima = calcular_metricas(test.values, previsoes_sarima)
print("\n📊 Métricas SARIMA:")
for metrica, valor in metricas_sarima.items():
    print(f"   {metrica}: {valor}")

# Visualização
plt.figure(figsize=(20, 6))
plt.plot(range(len(train)), train.values, label='Treino', linewidth=2)
plt.plot(range(len(train), len(train) + len(test)), test.values, 
         label='Teste (Real)', marker='o', linewidth=2, markersize=8)
plt.plot(range(len(train), len(train) + len(test)), previsoes_sarima, 
         label=f'Previsão SARIMA{melhor_ordem_sarima[0]}x{melhor_ordem_sarima[1]}', 
         marker='x', linewidth=2, markersize=10, linestyle='--')
plt.axvline(x=len(train), color='red', linestyle=':', linewidth=2, label='Início do Teste')
plt.title(f'Previsão SARIMA{melhor_ordem_sarima[0]}x{melhor_ordem_sarima[1]}', 
          fontsize=14, fontweight='bold')
plt.xlabel('Tempo')
plt.ylabel('Valor')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 13. Comparação ARIMA vs SARIMA

In [ ]:
# Comparação de métricas
comparacao = pd.DataFrame([
    {'Modelo': f'ARIMA{melhor_ordem}', **metricas_arima},
    {'Modelo': f'SARIMA{melhor_ordem_sarima[0]}x{melhor_ordem_sarima[1]}', **metricas_sarima}
])

print("\n" + "="*70)
print("  COMPARAÇÃO FINAL: ARIMA vs SARIMA")
print("="*70)
display(comparacao)
print("="*70)

# Identificar melhor modelo
melhor_idx = comparacao['MAE'].idxmin()
melhor = comparacao.loc[melhor_idx]
print(f"\n🏆 MELHOR MODELO: {melhor['Modelo']}")
print(f"   MAE: {melhor['MAE']}")
print(f"   MAPE: {melhor['MAPE']}%")

# Visualização comparativa
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# Gráfico 1: Previsões lado a lado
axes[0].plot(range(len(train), len(train) + len(test)), test.values, 
             label='Real', marker='o', linewidth=3, markersize=8)
axes[0].plot(range(len(train), len(train) + len(test)), previsoes_arima, 
             label=f'ARIMA{melhor_ordem}', marker='s', linewidth=2, 
             markersize=6, linestyle='--', alpha=0.7)
axes[0].plot(range(len(train), len(train) + len(test)), previsoes_sarima, 
             label=f'SARIMA{melhor_ordem_sarima[0]}x{melhor_ordem_sarima[1]}', 
             marker='^', linewidth=2, markersize=6, linestyle='--', alpha=0.7)
axes[0].set_title('Comparação de Previsões', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tempo')
axes[0].set_ylabel('Valor')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Comparação de métricas
comparacao.set_index('Modelo')[['MAE', 'RMSE']].plot(kind='bar', ax=axes[1])
axes[1].set_title('Comparação de Erros', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Erro')
axes[1].set_xlabel('')
axes[1].legend(['MAE', 'RMSE'])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 14. Exportar Resultados

In [ ]:
# Salvar previsões
df_previsoes = pd.DataFrame({
    'valor_real': test.values,
    'previsao_arima': previsoes_arima,
    'previsao_sarima': previsoes_sarima
})

output_path = '/Users/mayumi.castiglioni/Downloads/previsoes_arima_sarima.csv'
df_previsoes.to_csv(output_path, index=False)
print(f"✅ Previsões salvas em: {output_path}")

# Salvar comparação de modelos
comparacao_path = '/Users/mayumi.castiglioni/Downloads/comparacao_modelos.csv'
comparacao.to_csv(comparacao_path, index=False)
print(f"✅ Comparação salva em: {comparacao_path}")

# Salvar histórico de busca
historico_path = '/Users/mayumi.castiglioni/Downloads/historico_busca_modelos.xlsx'
with pd.ExcelWriter(historico_path) as writer:
    df_hist_arima.to_excel(writer, sheet_name='ARIMA', index=False)
    df_hist_sarima.to_excel(writer, sheet_name='SARIMA', index=False)
print(f"✅ Histórico de busca salvo em: {historico_path}")

print("\n🎉 Pipeline completa! Todos os resultados foram salvos.")